<a href="https://colab.research.google.com/github/khushitrivedi61/Machine-Learning/blob/main/practical1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.decomposition import PCA

In [12]:
df = pd.read_csv('/content/auto-mpg (1).csv')

display(df.head())

,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin,car name
0,18.0,8,307.0,130,3504,12.0,70,1,chevrolet chevelle malibu
1,15.0,8,350.0,165,3693,11.5,70,1,buick skylark 320
2,18.0,8,318.0,150,3436,11.0,70,1,plymouth satellite
3,16.0,8,304.0,150,3433,12.0,70,1,amc rebel sst
4,17.0,8,302.0,140,3449,10.5,70,1,ford torino


In [7]:
print(df.dtypes)

mpg             float64
cylinders         int64
displacement    float64
horsepower       object
weight            int64
acceleration    float64
model year        int64
origin            int64
car name         object
dtype: object


In [11]:
df.select_dtypes(include=np.number).columns

Index(['mpg', 'cylinders', 'displacement', 'weight', 'acceleration',
       'model year', 'origin'],
      dtype='object')

In [13]:
# To automatically find categorical columns:
categorical_columns = df.select_dtypes(include="object").columns

print("Categorical Columns:")
print(categorical_columns)

Categorical Columns:
Index(['horsepower', 'car name'], dtype='object')


In [14]:
df["origin"] = df["origin"].astype("object")

In [16]:
# To automatically find numeric columns:
numeric_columns = df.select_dtypes(include=np.number).columns

print("Numeric Columns:")
print(numeric_columns)

# origin is often represented by numbers such as 1, 2, 3,
# so technically pandas may identify it as numeric
# even though it represents categories.

Numeric Columns:
Index(['mpg', 'cylinders', 'displacement', 'weight', 'acceleration',
       'model year'],
      dtype='object')


In [18]:
print(df.isnull().sum())

#auto MPG commonly represents missing horsepower values using ?
#so first covert ? into missing value:

mpg             0
cylinders       0
displacement    0
horsepower      0
weight          0
acceleration    0
model year      0
origin          0
car name        0
dtype: int64


In [21]:
df["horsepower"] = pd.to_numeric(df["horsepower"], errors="coerce")
#convert horsepower into numeric

In [22]:
df["horsepower"]=df["horsepower"].fillna(df["horsepower"])

In [23]:
scaler = MinMaxScaler().set_output(transform="pandas")
scaled_data = scaler.fit_transform(df[numeric_columns])
scaled_data.head()

,mpg,cylinders,displacement,weight,acceleration,model year
0,0.239362,1.0,0.617571,0.536150,0.238095,0.0
1,0.159574,1.0,0.728682,0.589736,0.208333,0.0
2,0.239362,1.0,0.645995,0.516870,0.178571,0.0
3,0.186170,1.0,0.609819,0.516019,0.238095,0.0
4,0.212766,1.0,0.604651,0.520556,0.148810,0.0


In [25]:
categorial_columns =["origin", "car name"]

In [26]:
encoder=OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
).set_output(transform="pandas")

encoded_data = encoder.fit_transform(df[categorical_columns])
encoded_data.head()


,horsepower_46.0,horsepower_48.0,horsepower_49.0,horsepower_52.0,horsepower_53.0,horsepower_54.0,horsepower_58.0,horsepower_60.0,horsepower_61.0,horsepower_62.0,...,car name_volvo 145e (sw),car name_volvo 244dl,car name_volvo 245,car name_volvo 264gl,car name_volvo diesel,car name_vw dasher (diesel),car name_vw pickup,car name_vw rabbit,car name_vw rabbit c (diesel),car name_vw rabbit custom
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [27]:
#suppose mpg ia not target
X = df.drop("mpg", axis=1)
y =df["mpg"]

In [32]:
# Convert horsepower to numeric
df["horsepower"] = pd.to_numeric(df["horsepower"], errors="coerce")

# Fill missing horsepower values with the mean
df["horsepower"] = df["horsepower"].fillna(df["horsepower"].mean())

# Recreate X from the updated df to include the cleaned horsepower values
X = df.drop("mpg", axis=1)

# Select numeric columns
X_numeric = X.select_dtypes(include=np.number)

# Select best 3 features
selector = SelectKBest(
    score_func=f_regression,
    k=3
).set_output(transform="pandas")

X_selected = selector.fit_transform(X_numeric, y)

print(X_selected.head())

   cylinders  displacement  weight
0          8         307.0    3504
1          8         350.0    3693
2          8         318.0    3436
3          8         304.0    3433
4          8         302.0    3449
